# Does LightGBM's cross-conformal really cost more wall-clock than TabPFN's?

### What this is about, in one paragraph

A fraud model outputs a probability. A risk committee needs a guarantee: what
fraction of fraud will this catch, and can that be proven? *Conformal prediction*
converts the first into the second, by using a set of labelled examples to
calibrate a threshold with a distribution-free coverage guarantee. The catch is
that the standard method, *split conformal*, must hold out half of those labelled
examples to calibrate, and in fraud the scarce resource is confirmed positives,
not data and not compute. *Cross-conformal* spends none of them, because every row
is scored by a model that did not see it. Nobody uses it, because it costs K
refits. **TabPFN has no training step**, so K refits are K forward passes, which is
the observation the project [tabpfn-conformal](https://github.com/ilyas-elm/tabpfn-conformal)
is built on.

### The specific question here

That project registered five predictions before running anything, and reports four
of them as falsified, including two of its own about cost. The fifth, **P5**, is
the one it still reports as unsettled, and this notebook is what settles it.

P5 predicted that LightGBM's cross-conformal would cost far more wall-clock than
TabPFN's. The measurement said the opposite, but could not be trusted: TabPFN ran
through the Prior Labs API on their GPUs, while LightGBM ran on a laptop CPU. That
is not a race, since most of the TabPFN number was network round-trip. Every
wall-clock row in the project's baseline experiment is tagged
`wallclock_comparable: false` for that reason, and the repository declines to quote
the figure rather than resting a claim on it.

**What this does.** Both models, one machine, one accelerator, using TabPFN's
downloadable weights so that nothing crosses the network while the clock runs. The
protocol is otherwise unchanged: matched label budgets, the same pool and
evaluation construction, 2 budgets x 3 seeds x 2 families x 2 strategies, on Bank
Account Fraud (Jesus et al., NeurIPS 2022).

**What it cannot do.** It measures *local* TabPFN, not the managed API, so these
timings do not reproduce the API numbers and are not meant to. The gradient-fit
count is the hardware-independent comparison and does not move either way: 0 for
TabPFN, 1 for LightGBM split, 6 for LightGBM cross at K=5.

> **Reading this without running it.** Every cell's output is saved, so the
> measurement and the verdict are visible below without executing anything.
> Pressing *Copy & Edit* and running it needs a GPU and a TabPFN key, and if
> either is missing the notebook says so and recomputes the verdict from the
> committed results instead of failing.

## The hardware

The entire point of this run is that both models share one accelerator, so the
machine is recorded here rather than asserted.

In [ ]:
import subprocess, sys

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
HAS_GPU = bool(gpu)
print("gpu:   ", gpu or "none visible")
print("python:", sys.version.split()[0])

## Credentials and data

TabPFN asks for a one-time licence acceptance before it will download weights for
local use, and outside an interactive terminal it reads an API key from the
environment instead of prompting. The key **authorises the download and nothing
else**: inference runs on those weights, on this machine, which is what keeps the
measurement clean. It is read from a Kaggle secret, so it is never stored in the
notebook.

Nothing here raises. A reader without a key or without the dataset attached still
gets the verdict, recomputed further down from results committed to the
repository.

In [ ]:
import os, pathlib

HAS_TOKEN = False
try:
    from kaggle_secrets import UserSecretsClient
    for label in ("tabpfn v3.5", "TABPFN_TOKEN", "tabpfn_token", "conform"):
        try:
            os.environ["TABPFN_TOKEN"] = UserSecretsClient().get_secret(label)
            HAS_TOKEN, found = True, label
            break
        except Exception:
            continue
except ImportError:
    HAS_TOKEN = bool(os.environ.get("TABPFN_TOKEN"))   # running outside Kaggle
    found = "TABPFN_TOKEN from the environment"

print("token: ", found if HAS_TOKEN else "none (see the appendix)")

base = next(pathlib.Path("/kaggle/input").rglob("Base.csv"), None)
DATA = str(base.parent) if base else None
print("data:  ", DATA or "not attached (see the appendix)")

MEASURE = HAS_GPU and HAS_TOKEN and DATA is not None
print("\nwill run the measurement:", MEASURE)

## The code being measured

Cloned from the repository rather than pasted into a cell, so this notebook cannot
drift from the library it claims to be measuring. `experiments/kaggle/wallclock.py`
is about 200 lines and readable there.

In [ ]:
%cd /kaggle/working
!pip install -q tabpfn lightgbm
!rm -rf /kaggle/working/tabpfn-conformal
!git clone -q https://github.com/ilyas-elm/tabpfn-conformal.git
%cd /kaggle/working/tabpfn-conformal
!pip install -q -e .

## The measurement

Both families are warmed up before anything is timed. TabPFN does not load its
weights until `fit`, and on a fresh machine that first call also *downloads* them,
876 MB here. Timing it would have charged TabPFN a large network cost in the one
experiment whose entire purpose is to have no network in it.

Results are written after every configuration rather than at the end, so an
interrupted session still leaves usable rows. Roughly half an hour on a T4.

In [ ]:
if MEASURE:
    subprocess.run([sys.executable, "experiments/kaggle/wallclock.py",
                    "--data", DATA], check=True)
else:
    missing = [n for n, ok in (("a GPU", HAS_GPU), ("a TabPFN key", HAS_TOKEN),
                               ("the dataset", DATA is not None)) if not ok]
    print("Measurement skipped. This session is missing: " + ", ".join(missing) + ".")
    print("The appendix at the bottom says how to supply them.")
    print("The verdict below is recomputed from the results committed to the")
    print("repository, which is the run this notebook originally performed.")

## The verdict

The two families are run on the same seeds, so the comparison is paired rather
than a difference of means, and it reports the standard error and the number of
seeds behind it. A gap smaller than two standard errors is reported as no
separation, not as a win.

This reads `results/kaggle_wallclock.json`, whether it was just produced above or
came with the repository, and it states which machine produced it. If that machine
was a CPU, it refuses to draw a conclusion.

In [ ]:
subprocess.run([sys.executable, "experiments/analyze_kaggle.py"])

## Reading this

Whatever the timings say, the claim the project leads with is the
hardware-independent one: **0 gradient-trained fits against LightGBM's 6** at K=5.
That is why K-fold cross-conformal is affordable on TabPFN at all, and no choice
of machine changes it.

Wall-clock is the secondary question, and it is the one this notebook exists to
answer honestly rather than to win. P5 predicted TabPFN would come out ahead. The
earlier measurement suggested the opposite but was confounded, and this run is
what replaces it, whichever way it falls.

Method, benchmarks and the full falsification table:
<https://github.com/ilyas-elm/tabpfn-conformal>

---

## Appendix: running this, rather than reading it

*Copy & Edit* runs it on Kaggle's hardware. Four things have to be set, and the
notebook reports which are missing rather than failing.

1. **Accelerator**: GPU T4 x2, under Session options. On CPU the run settles
   nothing and the analysis refuses to draw a conclusion.
2. **Internet**: on, under Session options. It is off by default, and the install
   and the clone both need it. This and the GPU both require Kaggle phone
   verification.
3. **A TabPFN API key**, as a Kaggle secret attached to the notebook, from
   <https://ux.priorlabs.ai/account>, with the licence accepted on the Licenses
   tab of that site. The credentials cell tries a few likely secret labels and
   prints the one it found; a different label can be added to that list.
4. **The dataset**: Add Input, `Bank Account Fraud Dataset NeurIPS 2022`. The
   folder name does not matter, the notebook searches for the file.

Outside Kaggle it needs the same two packages, `TABPFN_TOKEN` in the environment,
and a directory holding `Base.csv`.

Without any of that, the repository's committed results still reproduce the whole
analysis on a laptop with no GPU and no key:

```bash
git clone https://github.com/ilyas-elm/tabpfn-conformal.git
cd tabpfn-conformal && pip install -e .
python experiments/analyze_kaggle.py
```